# Full Pipeline: coronagraphoto + photomancy.postproc

This notebook demonstrates the complete workflow from image simulation to SNR calculation:

1. **coronagraphoto** generates simulated coronagraphic observations
2. **photomancy.postproc** performs PSF subtraction and SNR analysis

A Mawet estimator can generate a full 2D statistic map for visualization or evaluate SNR at known target positions.

In [ ]:
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

import photomancy.postproc as cz

# Absolute Julian Dates need float64 precision.
jax.config.update("jax_enable_x64", True)

## Performance for Known Positions and Full Maps

The first call for each input shape includes JAX compilation. The table counts the positions evaluated after filtering the image.

| Method | Use Case | Complexity | Best For |
|--------|----------|------------|----------|
| `estimator(image, positions)` | Known positions | O(K) | Yield sims, pipelines |
| `estimator.map(image)` | Full 2D map | O(N²) | Visualization, blind searches |

In [ ]:
# Create a test image for timing comparison
test_image = jnp.zeros((300, 300))
test_positions = jnp.array([[150.0, 150.0], [100.0, 100.0], [200.0, 200.0]])
test_estimator = cz.DetectionEstimator.mawet(fwhm=4.0)

# Time the position call - first call includes compilation
t0 = time.perf_counter()
_ = test_estimator(test_image, test_positions).statistic.block_until_ready()
snr_compile_time = time.perf_counter() - t0

# Time the position call - reuse (after compilation)
t0 = time.perf_counter()
for _ in range(100):
    _ = test_estimator(test_image, test_positions).statistic.block_until_ready()
snr_reuse_time = (time.perf_counter() - t0) / 100

# Time the map call - first call includes compilation
t0 = time.perf_counter()
_ = test_estimator.map(test_image).block_until_ready()
snr_map_compile_time = time.perf_counter() - t0

# Time the map call - reuse (after compilation)
t0 = time.perf_counter()
for _ in range(10):
    _ = test_estimator.map(test_image).block_until_ready()
snr_map_reuse_time = (time.perf_counter() - t0) / 10

print("Performance Comparison (300x300 image, 3 positions):")
print("\n3 known positions:")
print(f"  First call (compile): {snr_compile_time * 1000:.0f} ms")
print(f"  Subsequent calls:     {snr_reuse_time * 1000:.2f} ms")
print("\nFull 90,000-pixel map:")
print(f"  First call (compile): {snr_map_compile_time * 1000:.0f} ms")
print(f"  Subsequent calls:     {snr_map_reuse_time * 1000:.0f} ms")
print(
    f"\nSpeedup: {snr_map_reuse_time / snr_reuse_time:.0f}x faster for known positions"
)

## 1. Download Example Data

The example data comes through `pooch`. The first time you run this, it will download:
- A coronagraph YIP (eac1_aavc_512 created by Susan Redmond)
- An ExoVista scene (modified Solar System)

In [ ]:
import logging

# Fetch example data (downloads from GitHub if not cached)
from skyscapes.datasets import fetch_scene
from yippy import open_yip

# The loader reports header details at INFO and WARNING level; show errors only.
logging.getLogger("yippy").setLevel(logging.ERROR)

package = open_yip("eac1_aavc_2d")
scene_path = fetch_scene()

print(f"Coronagraph: {package.source.path}")
print(f"Scene: {scene_path}")

## 2. Load Data with skyscapes and yippy

skyscapes reads the ExoVista file into a `Scene`, and yippy provides the coronagraph.

In [ ]:
from coronagraphoto import build_exposure, expected, observe
from hwoutils.conversions import arcsec_to_lambda_d
from optixstuff import (
    ConstantThroughput,
    FramePlan,
    IdealDetector,
    Instrument,
    SimplePrimary,
)
from optixstuff.spectral import Band
from skyscapes import from_exovista

# Load the scene (star and planets; the zodi background is not used below)
scene = from_exovista(scene_path, load_disk=False)
coronagraph = package.prepare()

n_planets = scene.n_planets
print(f"Loaded scene with {n_planets} planets")

## 3. Setup

In [ ]:
PIXEL_SCALE_ARCSEC = 1 / 512
WAVELENGTH_NM = 550.0
BIN_WIDTH_NM = 100.0
# The band must fit inside the product's 10% fractional width (55 nm at 550 nm).
BAND_WIDTH_NM = 50.0
# The coronagraph tables are computed for one telescope, and the instrument
# checks that the primary matches. Read the diameter from the tables.
DIAMETER_M = float(coronagraph.frame.diameter_m)

instrument = Instrument(
    primary=SimplePrimary(
        circumscribed_diameter_m=DIAMETER_M,
        collecting_area_m2=float(jnp.pi * (DIAMETER_M / 2) ** 2),
    ),
    attenuating_elements=(ConstantThroughput(throughput=0.9),),
    coronagraph=coronagraph,
    detector=IdealDetector(pixel_scale_arcsec=PIXEL_SCALE_ARCSEC, shape=(300, 300)),
)

# FWHM of an Airy disk is about 1.03 lambda/D. Convert the detector pixel
# scale to lambda/D at the observing wavelength to get the FWHM in pixels.
pixel_scale_lod = float(
    arcsec_to_lambda_d(PIXEL_SCALE_ARCSEC, WAVELENGTH_NM, DIAMETER_M)
)
fwhm = 1.03 / pixel_scale_lod

print(f"Detector pixel scale: {pixel_scale_lod:.4f} (lambda/D)/pixel")
print(f"Calculated FWHM: {fwhm:.2f} pixels")

## 4. Simulate Observation with coronagraphoto

One `Exposure` holds the epoch, the roll and the wavelength band. From it, coronagraphoto produces:
- A noisy observation (star leakage + planets + detector noise), from `observe`
- The noiseless expected rates, from `expected`, which keep the star and the planets in separate fields and so give a noiseless stellar model for perfect subtraction

The detector frame follows the FITS display convention. Row increases toward
detector +y (north at a roll of 0), and column increases
toward detector +x, which is **west**. With `imshow(origin="lower")`, north is
up and east is to the left.

In [ ]:
# Exposure: one day at the scene epoch, with no roll
exposure_time_s = 24 * 3600.0
exposure = build_exposure(
    scene,
    instrument,
    (Band(WAVELENGTH_NM - BAND_WIDTH_NM / 2, WAVELENGTH_NM + BAND_WIDTH_NM / 2),),
    time_d=0.0,
    roll_deg=0.0,
    plan=FramePlan(frame_time_s=exposure_time_s, n_frames=1),
)

# One draw of the full observation: star, planets and detector noise
image = observe(jax.random.PRNGKey(0), scene, instrument, exposure)
noisy_observation = image.data[0]

fig, ax = plt.subplots()
ax.imshow(noisy_observation, origin="lower", cmap="magma")
ax.set_title("Observation")
ax.set_xlabel("column (pixels, +column = west)")
ax.set_ylabel("row (pixels, +row = north)")
plt.show()

print(f"Observation shape: {noisy_observation.shape}")
print(f"Max signal: {float(jnp.max(noisy_observation)):.1f} e-")

## 5. PSF Subtraction with photomancy.postproc

With coronagraphoto, we have the noiseless stellar expectation - this enables perfect PSF subtraction.

In [ ]:
# Noiseless expected rates in electrons/s/pixel, one field per source
rates = expected(scene, instrument, exposure)

# Noiseless stellar expectation (counts) for perfect subtraction
star_expectation = rates.star_leakage[0] * exposure_time_s

# Perfect PSF subtraction
residual = cz.subtract_star(noisy_observation, star_expectation)

print(f"Residual mean: {float(jnp.mean(residual)):.2f} e-")
print(f"Residual std: {float(jnp.std(residual)):.2f} e-")

## 6. SNR Analysis: Method 1 - SNR Map

Use `estimator.map(image)` to generate a statistic at every pixel for:
- Visualization
- Blind searches
- Understanding the detection landscape

In [ ]:
# Generate full SNR detection map
estimator = cz.DetectionEstimator.mawet(fwhm)
snr_detection_map = estimator.map(residual)

print(f"SNR map shape: {snr_detection_map.shape}")
print(f"Max SNR in map: {float(jnp.nanmax(snr_detection_map)):.1f}")

In [ ]:
# Plot the observation, residual, and SNR map side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Raw observation
im0 = axes[0].imshow(noisy_observation, origin="lower", cmap="magma")
axes[0].set_title("Noisy Observation", fontsize=12, fontweight="bold")
axes[0].set_xlabel("column (pixels, +column = west)")
axes[0].set_ylabel("row (pixels, +row = north)")
plt.colorbar(im0, ax=axes[0], label="e$^-$")

# Residual after PSF subtraction
vmax = float(jnp.nanpercentile(jnp.abs(residual), 99))
im1 = axes[1].imshow(residual, origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[1].set_title("Residual (after PSF subtraction)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("column (pixels, +column = west)")
plt.colorbar(im1, ax=axes[1], label="e$^-$")

# SNR detection map
im2 = axes[2].imshow(snr_detection_map, origin="lower", cmap="viridis", vmin=0, vmax=10)
axes[2].set_title("SNR Detection Map", fontsize=12, fontweight="bold")
axes[2].set_xlabel("column (pixels, +column = west)")
plt.colorbar(im2, ax=axes[2], label="SNR")

plt.tight_layout()
plt.show()

## 7. SNR Analysis: Method 2 - Known Positions

When you know the planet positions (e.g., from orbital predictions), the `snr` function is much faster.

This is the preferred method for:
- Yield simulations
- Follow-up observations
- Performance-critical pipelines

Planet positions come from the orbit as sky offsets (east, north) in arcsec.
Detector pixels follow the FITS convention, so east is toward **decreasing**
column:

```text
column = center - east / pixel_scale
row    = center + north / pixel_scale
center = (N - 1) / 2
```

This holds at a telescope position angle of 0 (the general rotation is in the
coronagraphoto documentation).

In [ ]:
# Sky offsets (east, north) of every planet at the exposure epoch, in arcsec
planet_pos_arcsec = scene.positions(jnp.array([exposure.time_d]))[:, :, 0]
# (2, n_planets): row 0 east, row 1 north

east, north = planet_pos_arcsec
ny, nx = noisy_observation.shape
center_row, center_col = (ny - 1) / 2.0, (nx - 1) / 2.0

# FITS convention: +column = west, so east decreases the column index
planet_col = center_col - east / PIXEL_SCALE_ARCSEC
planet_row = center_row + north / PIXEL_SCALE_ARCSEC
planet_positions = jnp.stack([planet_row, planet_col], axis=1)  # (n_planets, 2)

in_frame = (
    (planet_row >= 0)
    & (planet_row <= ny - 1)
    & (planet_col >= 0)
    & (planet_col <= nx - 1)
)

print(f"Number of planets: {planet_positions.shape[0]}")
print("Planet positions (row, column):")
for i, pos in enumerate(planet_positions):
    note = "" if bool(in_frame[i]) else "  (outside the frame)"
    print(f"  Planet {i}: ({float(pos[0]):.1f}, {float(pos[1]):.1f}){note}")

In [ ]:
# Check the placement: every in-frame planet outside the coronagraph's inner
# working angle (taken here as 4 lambda/D) must peak at its predicted pixel in
# the noiseless planet-only count map. Closer planets are suppressed and have
# no clean peak.
separation_lod = jnp.hypot(east, north) / PIXEL_SCALE_ARCSEC * pixel_scale_lod
checkable = in_frame & (separation_lod > 4.0)

planet_map = rates.planet[0]

half = 3
n_checked = 0
for i in range(n_planets):
    if not bool(checkable[i]):
        continue
    r0, c0 = round(float(planet_row[i])), round(float(planet_col[i]))
    window = planet_map[r0 - half : r0 + half + 1, c0 - half : c0 + half + 1]
    dr, dc = jnp.unravel_index(jnp.argmax(window), window.shape)
    peak_row, peak_col = r0 - half + int(dr), c0 - half + int(dc)
    print(
        f"Planet {i}: predicted ({float(planet_row[i]):.1f},"
        f" {float(planet_col[i]):.1f}),"
        f" brightest pixel ({peak_row}, {peak_col})"
    )
    assert abs(peak_row - float(planet_row[i])) <= 1.0
    assert abs(peak_col - float(planet_col[i])) <= 1.0
    n_checked += 1

assert n_checked >= 2

In [ ]:
# Calculate SNR at known positions
snr_values = estimator(residual, planet_positions).statistic

print("\nSNR values at planet positions:")
for i, snr_val in enumerate(snr_values):
    print(f"  Planet {i}: SNR = {float(snr_val):.2f}")

## 8. Visualization: Comparing Both Methods

Let's overlay the known planet positions on the SNR map to compare the two approaches.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: SNR map with planet positions marked
im0 = axes[0].imshow(snr_detection_map, origin="lower", cmap="viridis", vmin=0, vmax=10)
for i, pos in enumerate(planet_positions):
    # Only show planets within the image bounds
    if 0 <= pos[0] < ny and 0 <= pos[1] < nx:
        circle = Circle(
            (float(pos[1]), float(pos[0])), fwhm, fill=False, color="red", linewidth=2
        )
        axes[0].add_patch(circle)
        axes[0].annotate(
            f"{i}",
            (float(pos[1]) + 5, float(pos[0]) + 5),
            color="white",
            fontsize=10,
            fontweight="bold",
        )
axes[0].set_title("SNR Map with Planet Positions", fontsize=12, fontweight="bold")
axes[0].set_xlabel("column (pixels, +column = west)")
axes[0].set_ylabel("row (pixels, +row = north)")
plt.colorbar(im0, ax=axes[0], label="SNR")

# Right: Bar chart comparing SNR values
# Filter to planets within the image
valid_indices = []
valid_snrs = []
for i, (pos, snr_val) in enumerate(zip(planet_positions, snr_values, strict=False)):
    if 0 <= pos[0] < ny and 0 <= pos[1] < nx and bool(jnp.isfinite(snr_val)):
        valid_indices.append(i)
        valid_snrs.append(float(snr_val))

colors = ["green" if s >= 5 else "orange" if s >= 3 else "red" for s in valid_snrs]
bars = axes[1].bar([f"Planet {i}" for i in valid_indices], valid_snrs, color=colors)
axes[1].axhline(
    y=5, color="green", linestyle="--", linewidth=2, label="5 sigma detection"
)
axes[1].axhline(
    y=3, color="orange", linestyle="--", linewidth=2, label="3 sigma threshold"
)
axes[1].set_ylabel("SNR", fontsize=11)
axes[1].set_title("SNR at Known Planet Positions", fontsize=12, fontweight="bold")
axes[1].legend(loc="upper right")
axes[1].set_ylim(0, max(valid_snrs) * 1.2 if valid_snrs else 10)

plt.tight_layout()
plt.show()

## Summary

Both methods use the same underlying Mawet et al. (2014) small-sample statistics, so they produce identical results at the same positions.

Pass candidate coordinates to the estimator for known positions in yield simulations or performance-critical pipelines.

Use `estimator.map(image)` for visualization and blind searches.